In [1]:
# %% Setup
import os
import time
 
import pandas as pd
from dotenv import load_dotenv
from entsoe import EntsoePandasClient
 
load_dotenv()
client = EntsoePandasClient(api_key=os.environ["ENTSOE_API_KEY"])
 
TZ = "Europe/Rome"
ZONE = "IT_NORD"

In [17]:
def day(date_str):
    """Return a (start, end) pair covering one calendar day.

    DateOffset is calendar-aware; Timedelta is an absolute 24 hours.
    On the October DST day the calendar day is 25 hours and Timedelta
    falls an hour short; on the March day it is 23 hours and Timedelta
    reaches an hour into the next day.
    """
    start = pd.Timestamp(date_str, tz=TZ)
    return start, start + pd.DateOffset(days=1)

In [3]:
def fetch(query, zone, date_str):
    """Fetch one day, retrying the transient 599 the API sometimes returns.
 
    A NoMatchingDataError means 'nothing published' - that is a real answer,
    so it is raised straight away instead of being retried.
    """
    start, end = day(date_str)
    for attempt in range(3):
        try:
            return query(zone, start=start, end=end)
        except Exception as exc:
            if type(exc).__name__ == "NoMatchingDataError":
                raise
            if attempt == 2:
                raise
            print(f"    {type(exc).__name__}, retrying")
            time.sleep(2)
        finally:
            time.sleep(0.3)

In [4]:
def step_of(result):
    """The most common gap between timestamps - the real resolution."""
    return result.index.to_series().diff().value_counts().index[0]

In [5]:
# %% Gap 1a - load, one day per month
months = pd.period_range("2024-06", "2025-06", freq="M")
 
for month in months:
    date_str = f"{month}-15"
    result = fetch(client.query_load, ZONE, date_str)
    print(f"{date_str}   rows={len(result):>3}   step={step_of(result)}")

2024-06-15   rows= 24   step=0 days 01:00:00
2024-07-15   rows= 24   step=0 days 01:00:00
2024-08-15   rows= 24   step=0 days 01:00:00
2024-09-15   rows= 24   step=0 days 01:00:00
2024-10-15   rows= 24   step=0 days 01:00:00
2024-11-15   rows= 24   step=0 days 01:00:00
2024-12-15   rows= 24   step=0 days 01:00:00
2025-01-15   rows= 96   step=0 days 00:15:00
2025-02-15   rows= 96   step=0 days 00:15:00
2025-03-15   rows= 96   step=0 days 00:15:00
2025-04-15   rows= 96   step=0 days 00:15:00
2025-05-15   rows= 96   step=0 days 00:15:00
2025-06-15   rows= 96   step=0 days 00:15:00


In [6]:
for month in months:
    date_str = f"{month}-15"
    result = fetch(client.query_generation, ZONE, date_str)
    print(f"{date_str}   rows={len(result):>3}   step={step_of(result)}")

2024-06-15   rows= 24   step=0 days 01:00:00
2024-07-15   rows= 24   step=0 days 01:00:00
2024-08-15   rows= 24   step=0 days 01:00:00
2024-09-15   rows= 24   step=0 days 01:00:00
2024-10-15   rows= 24   step=0 days 01:00:00
2024-11-15   rows= 24   step=0 days 01:00:00
2024-12-15   rows= 24   step=0 days 01:00:00
2025-01-15   rows= 96   step=0 days 00:15:00
2025-02-15   rows= 96   step=0 days 00:15:00
2025-03-15   rows= 96   step=0 days 00:15:00
2025-04-15   rows= 96   step=0 days 00:15:00
2025-05-15   rows= 96   step=0 days 00:15:00
2025-06-15   rows= 96   step=0 days 00:15:00


In [8]:
SCAN_MONTH = "2024-12"
 
for date in pd.period_range(f"{SCAN_MONTH}-01", freq="D", periods=31):
    if str(date)[:7] != SCAN_MONTH:
        break
    result = fetch(client.query_load, ZONE, str(date))
    print(f"{date}   rows={len(result):>3}   step={step_of(result)}")

2024-12-01   rows= 24   step=0 days 01:00:00
2024-12-02   rows= 24   step=0 days 01:00:00
2024-12-03   rows= 24   step=0 days 01:00:00
2024-12-04   rows= 24   step=0 days 01:00:00
2024-12-05   rows= 24   step=0 days 01:00:00
2024-12-06   rows= 24   step=0 days 01:00:00
2024-12-07   rows= 24   step=0 days 01:00:00
2024-12-08   rows= 24   step=0 days 01:00:00
2024-12-09   rows= 24   step=0 days 01:00:00
2024-12-10   rows= 24   step=0 days 01:00:00
2024-12-11   rows= 24   step=0 days 01:00:00
2024-12-12   rows= 24   step=0 days 01:00:00
2024-12-13   rows= 24   step=0 days 01:00:00
2024-12-14   rows= 24   step=0 days 01:00:00
2024-12-15   rows= 24   step=0 days 01:00:00
2024-12-16   rows= 24   step=0 days 01:00:00
2024-12-17   rows= 24   step=0 days 01:00:00
2024-12-18   rows= 24   step=0 days 01:00:00
2024-12-19   rows= 24   step=0 days 01:00:00
2024-12-20   rows= 24   step=0 days 01:00:00
2024-12-21   rows= 24   step=0 days 01:00:00
2024-12-22   rows= 24   step=0 days 01:00:00
2024-12-23

In [18]:
# ## Gap 2 - the missing hour on the October DST day
# %% Gap 2a - how much time does the data actually cover
dst = fetch(client.query_day_ahead_prices, ZONE, "2025-10-26")
 
span_hours = (dst.index[-1] - dst.index[0]).total_seconds() / 3600
 
print("rows         :", len(dst))
print("first        :", dst.index[0])
print("last         :", dst.index[-1])
print("span         :", round(span_hours, 2), "h")
print("expected rows:", int(span_hours * 4) + 1)

rows         : 101
first        : 2025-10-26 00:00:00+02:00
last         : 2025-10-27 00:00:00+01:00
span         : 25.0 h
expected rows: 101


In [10]:
# %% Gap 2b - does the index have holes
gaps = dst.index.to_series().diff()
irregular = gaps[gaps.notna() & (gaps != pd.Timedelta(minutes=15))]
 
if len(irregular) == 0:
    print("no holes - every interval is 15 minutes apart")
    print("so the missing hour is at one of the ends, not in the middle")
else:
    print("gaps that are not 15 minutes:")
    print(irregular.to_string())

no holes - every interval is 15 minutes apart
so the missing hour is at one of the ends, not in the middle


In [11]:
# %% Gap 2c - the transition, in local time and UTC
# Local time repeats 02:00-03:00 on this day; UTC does not. Seeing both
# together shows whether the repeated hour was published once or twice.
window = dst.loc["2025-10-26 01:00":"2025-10-26 04:00"]
 
table = pd.DataFrame({
    "local": window.index.strftime("%H:%M"),
    "offset": [t.utcoffset() for t in window.index],
    "utc": window.index.tz_convert("UTC").strftime("%H:%M"),
    "price": window.values,
})
print(table.to_string(index=False))

local          offset   utc  price
01:00 0 days 02:00:00 23:00  94.87
01:15 0 days 02:00:00 23:15  89.77
01:30 0 days 02:00:00 23:30  90.90
01:45 0 days 02:00:00 23:45  84.06
02:00 0 days 02:00:00 00:00  92.83
02:15 0 days 02:00:00 00:15  90.90
02:30 0 days 02:00:00 00:30  84.09
02:45 0 days 02:00:00 00:45  75.34
02:00 0 days 01:00:00 01:00  82.34
02:15 0 days 01:00:00 01:15  90.78
02:30 0 days 01:00:00 01:30  89.92
02:45 0 days 01:00:00 01:45  80.00
03:00 0 days 01:00:00 02:00  81.20
03:15 0 days 01:00:00 02:15  78.31
03:30 0 days 01:00:00 02:30  75.88
03:45 0 days 01:00:00 02:45  73.00
04:00 0 days 01:00:00 03:00  78.90


In [15]:
# ## Gap 3 - when does IT_CALA start
# %% Gap 3 - IT_CALA, one day per month, get changed at 2021-01 month, now get more details
SCAN_MONTH = "2020-12"
 
for date in pd.period_range(f"{SCAN_MONTH}-01", freq="D", periods=31):
    if str(date)[:7] != SCAN_MONTH:
        break
    result = fetch(client.query_day_ahead_prices, "IT_CALA", str(date))
    print(f"{date}   rows={len(result):>3}   step={step_of(result)}")

NoMatchingDataError: 

In [19]:
# %% confirm - what does the switch day actually look like
switch_day = fetch(client.query_load, ZONE, "2025-01-01")
deltas = switch_day.index.to_series().diff()

print("delta counts:")
print(deltas.value_counts().to_string())
print("\nfirst 6 timestamps:")
print(switch_day.head(6).to_string())

delta counts:
0 days 00:15:00    95

first 6 timestamps:
                           Actual Load
2025-01-01 00:00:00+01:00      11528.0
2025-01-01 00:15:00+01:00      11695.0
2025-01-01 00:30:00+01:00      11553.0
2025-01-01 00:45:00+01:00      11473.0
2025-01-01 01:00:00+01:00      11288.0
2025-01-01 01:15:00+01:00      11192.0


In [20]:
# %% does the switch date hold for every zone
ZONES = ["IT_NORD", "IT_CNOR", "IT_CSUD", "IT_SUD",
         "IT_CALA", "IT_SICI", "IT_SARD"]

for zone in ZONES:
    before = fetch(client.query_load, zone, "2024-12-30")
    after = fetch(client.query_load, zone, "2024-12-31")
    print(f"{zone:<10} 12-30: rows={len(before):>3} {step_of(before)}   "
          f"12-31: rows={len(after):>3} {step_of(after)}")

IT_NORD    12-30: rows= 24 0 days 01:00:00   12-31: rows= 93 0 days 00:15:00
IT_CNOR    12-30: rows= 24 0 days 01:00:00   12-31: rows= 93 0 days 00:15:00
IT_CSUD    12-30: rows= 24 0 days 01:00:00   12-31: rows= 93 0 days 00:15:00
IT_SUD     12-30: rows= 24 0 days 01:00:00   12-31: rows= 93 0 days 00:15:00
IT_CALA    12-30: rows= 24 0 days 01:00:00   12-31: rows= 93 0 days 00:15:00
IT_SICI    12-30: rows= 24 0 days 01:00:00   12-31: rows= 93 0 days 00:15:00
IT_SARD    12-30: rows= 24 0 days 01:00:00   12-31: rows= 93 0 days 00:15:00


In [21]:
# %% does the DST row count hold for every zone
for zone in ZONES:
    result = fetch(client.query_day_ahead_prices, zone, "2025-10-26")
    print(f"{zone:<10} rows={len(result)}")

IT_NORD    rows=101
IT_CNOR    rows=101
IT_CSUD    rows=101
IT_SUD     rows=101
IT_CALA    rows=101
IT_SICI    rows=101
IT_SARD    rows=101
